# Sesión 3: Fundamentos Prácticos de Redes Neuronales Artificiales (MLP)

En este cuaderno pondremos en práctica todos los conceptos matemáticos expuestos en clase, utilizando **estándares de código reales de la industria**. Cada bloque de código está diseñado para ser estudiado paso a paso con máxima exhaustividad en los comentarios.

Implementaremos una auténtica Red Neuronal Profunda (una capa de entrada, una capa oculta "Hidden" pura, y una capa de salida) en tres niveles de abstracción computacional distintos:

1. **Puro NumPy:** La pura matemática detrás de la red. Crearemos la Transformación Afín ($z=wx+b$), la no-linealidad *ReLU*, y la Regla de la Cadena (*Backpropagation*) **a mano** cruzando las tres profundidades.
2. **PyTorch Bajo Nivel:** Le cederemos el cálculo de derivadas al motor `Autograd` de PyTorch, pero gestionaremos los tensores y el descenso estocástico nosotros.
3. **PyTorch Estado-Del-Arte (`nn.Sequential`):** La forma profesional en que se construyen los modelos profundos en la industria real.

In [ ]:
# Matemáticas base y visualización
import numpy as np
import matplotlib.pyplot as plt

# PyTorch core y abstracciones SOTA de la industria
import torch
import torch.nn as nn
import torch.optim as optim

plt.style.use('dark_background')

--- 
## 0. El Dataset de Prueba (Regresión No-Lineal)

Generaremos un problema que sea **imposible** de resolver para una regresión lineal plana clásica: una curva senoidal paramétrica.

In [ ]:
# FIJAMOS LA SEMILLA (Seed):
# Asegura determinismo: si entrenamos esto en otra máquina, los pesos aleatorios inician igual.
np.random.seed(33)
torch.manual_seed(33)

# GENERACIÓN DE DATOS (Features: X):
# Creamos 200 puntos equitativos entre -3 y +3. 
# El .reshape(-1, 1) transforma el vector plano en una verdadera Columna (Shape: [200, 1])
X = np.linspace(-3, 3, 200).reshape(-1, 1)

# GENERACIÓN DE RESPUESTA REAL (Target / Ground Truth: y):
# La respuesta es la onda SENO contaminada con variables aleatorias (ruido Gaussiano).
y = np.sin(X) + np.random.randn(200, 1) * 0.1

print("Estructura Creada:")
print(f"-> Tensor X (Entradas): {X.shape} | 200 registros de datos.")
print(f"-> Tensor y (Objetivo): {y.shape} | 200 etiquetas continuas de Ground Truth.")

# Inspección Visual
plt.figure(figsize=(8, 4))
plt.scatter(X, y, color='cyan', alpha=0.6, label='Dataset Toy')
plt.title('Problema Matemático a Modelar')
plt.legend()
plt.show()

---
## 1. El Laboratorio Matemático (NumPy Desde Cero)

A continuación vamos a escribir la clase fundamental del perceptrón. Cualquier capa poseerá dos métodos vitales:
- `forward()`: Ingiere `X`, aplica transformación matricial y escupe logits `Z` (o activaciones `A`).
- `backward()`: Recibe el gradiente de error (`dZ` o `dA`) que viaja desde el final y aplica el cálculo diferencial (Regla de la Cadena) reescribiendo sus propios tensores (`W` y `b`).

In [ ]:
# ==================================================================
# 1.1 LA CAPA DENSA (El "Cerebro" y la Transformación Afín)
# ==================================================================
class Linear:
    def __init__(self, in_features, out_features, name="Linear"):
        self.name = name
        
        # INICIALIZACIÓN DE PESOS (Weights - W):
        # Seguimos distribución normal estándar de Numpy.
        # Multiplicamos por 0.1 para que arranque prevenida contra 'Gradientes Explosivos'.
        self.W = np.random.randn(in_features, out_features) * 0.1
        
        # INICIALIZACIÓN DEL SESGO (Bias - b):
        # El Bias es de índole estática. Siempre arranca limpio en valor 0.
        self.b = np.zeros((1, out_features)) 
        
    def forward(self, X, verbose=False):
        # CACHÉ (Memoria Estocástica):
        # Almacenamos 'X' original internamente. Es OBLIGATORIO recordarlo 
        # para la Regla de la Cadena durante el paso de Backpropagation.
        self.X = X 
        
        # ECUACIÓN LINEAL MATEMÁTICA PURA (Z = X * W + b):
        # El operador '@' denota Dot Product (Multiplicación de Matrices masiva).
        Z = self.X @ self.W + self.b
        
        if verbose:
            print(f"[{self.name}] FORWARD: Entrada {X.shape}  -->  Multiplicamos por Pesos {self.W.shape}  -->  Salida Pura (Z) {Z.shape}")
            
        return Z
        
    def backward(self, dZ, lr, verbose=False):
        # BATCH SIZE (Lote m):
        # Extraemos cuántos ejemplos fluyeron en total para poder promediar los castigos matriciales.
        m = self.X.shape[0]
        
        # DERIVADAS PARCIALES MATRICIALES (Regla de la Cadena):
        # -> Derivada del Factor Respecto a los Pesos (dW): X_Traspuesta @ Culpa(dZ)
        # -> Derivada del Factor Respecto al Sesgo (db): Suma de Culpa (dZ) vertical (axis=0)
        dW = (self.X.T @ dZ) / m
        db = np.sum(dZ, axis=0, keepdims=True) / m
        
        # ERROR TRASPASABLE AL NODO ANTERIOR (dX):
        # Calculamos la propagación paramétrica (el residuo) hacia la capa que nos precede
        dX = dZ @ self.W.T 
        
        if verbose:
            print(f"[{self.name}] BACKWARD: Culpa Recibida dZ {dZ.shape}  -->  Actualización Local dW {dW.shape}  -->  Traspasando Culpa Anterior dX {dX.shape}")

        # DESCENSO DE GRADIENTE (SGD Base):
        # Aplicamos la reducción a los propios parámetros del nodo.
        self.W -= lr * dW
        self.b -= lr * db
        
        return dX

# ==================================================================
# 1.2 INTERRUPTOR NO-LINEAL (La Función ReLU)
# ==================================================================
class ReLU:
    def __init__(self, name="ReLU"):
        self.name = name
        
    def forward(self, Z, verbose=False):
        self.Z = Z
        
        # APLICAR RELU (Rectified Linear Unit):
        # np.maximum devuelve 0 para números negativos, 
        # transfiere intactamente valores positivos -> Produce la "Activación (A)".
        A = np.maximum(0, Z) 
        
        if verbose:
            print(f"[{self.name}] FORWARD: Traga Logit Z {Z.shape}  -->  Truncado negativo (ReLU)  -->  Eyecta Activación A {A.shape}")
        return A
        
    def backward(self, dA, lr, verbose=False):
        dZ = dA.copy()
        
        # CÓMPUTO DIFERENCIAL RELU:
        # Su derivada matemática equivale a 1 si el punto era > 0, y 0 nulo si era <= 0.
        # Físicamente: Si la neurona estaba desactivada al evaluar, no contribuyó nunca al 
        # veredicto y no se le puede asignar ninguna porción de 'culpa' del error general.
        dZ[self.Z <= 0] = 0 
        
        if verbose:
            print(f"[{self.name}] BACKWARD: Propagando derivadas dA {dA.shape}  -->  Anula gradientes <0  -->  Exige dZ al predecesor {dZ.shape}")
        return dZ

# ==================================================================
# 1.3 EL JUEZ EVALUADOR (Receptor MSE de Pérdidas)
# ==================================================================
class MSELoss:
    @staticmethod
    def forward(y_pred, y):
        return np.mean((y_pred - y)**2)

    @staticmethod
    def backward(y_pred, y, verbose=False):
        # DERIVADA CLÁSICA INICIAL:
        # d( (y_pred - y)**2 ) = 2 * (y_pred - y)
        grad_inicial = 2 * (y_pred - y) / y.shape[0]
        
        if verbose:
            print(f"\n[Loss_Criterion] CHISPA INICIAL: Derivada Analítica MSE dPred {grad_inicial.shape} arrancando el túnel Backprop.\n")
        return grad_inicial

#### 1.4 Test Unilateral (Logging Explicativo)
Para ser rigurosos con la arquitectura MLP, crearemos lo que se llama una red paramétrica profunda: esto compone una capa enlazadora (Input), una capa **completamente Oculta / Hidden** (Capa que no tiene conexión alguna con la realidad de los features ni con las variables target), y la capa concentradora (Output).

Evaluaremos visualmente como las Shapes cruzan de `[1 -> 32]` luego de `[32 -> 32]` y al final de `[32 -> 1]`.

In [ ]:
# EMPAQUETADO SÍLICA COMPLETO: 
layer_input = Linear(in_features=1, out_features=32, name="Layer_Input")
relu1 = ReLU(name="ReLU_Input")

layer_hidden = Linear(in_features=32, out_features=32, name="Layer_HIDDEN_Oculta")
relu2 = ReLU(name="ReLU_Hidden")

layer_output = Linear(in_features=32, out_features=1, name="Layer_Output")

print("====== [ 1. FASE DE INFERENCIA FORWARD (Viaje De Señal Profundo) ] ======")
Z1 = layer_input.forward(X, verbose=True)
A1 = relu1.forward(Z1, verbose=True)

Z2 = layer_hidden.forward(A1, verbose=True)
A2 = relu2.forward(Z2, verbose=True)

y_pred_np = layer_output.forward(A2, verbose=True)

print("\n====== [ 2. EVALUACIÓN Y LOSS COMPUTACIONAL ] ======")
loss_val = MSELoss.forward(y_pred_np, y)
print(f"-> Coste Actualizado (MSE): Error aleatorio inicial: {loss_val:.5f}")

print("\n====== [ 3. RETROPROPAGACIÓN (Cascada Atrás Completa) ] ======")
lr_test = 0.02
loss_grad = MSELoss.backward(y_pred_np, y, verbose=True)

# Pasos sucesivos inverso
dA2 = layer_output.backward(loss_grad, lr=lr_test, verbose=True)

dZ2 = relu2.backward(dA2, lr=lr_test, verbose=True)
dA1 = layer_hidden.backward(dZ2, lr=lr_test, verbose=True)

dZ1 = relu1.backward(dA1, lr=lr_test, verbose=True)
dX_final = layer_input.backward(dZ1, lr=lr_test, verbose=True)

print("\n✅ ¡Pesos actualizados matemáticamente en todas las capas profundas!")

#### 1.5 Training Loop NumPy Definitivo
Dejamos correr el sistema ciegamente iterando de forma continua por iteraciones silenciosas de la red.

In [ ]:
epochs = 2000
history_numpy = []
lr = 0.02

print(f"-> Loop Autónomo NumPy Integrado: Evaluando {epochs} Epochs...")
for epoch in range(epochs):
    # -- 1. Fase Silenciosa Matriz (Inferencia Forward) --
    Z1 = layer_input.forward(X)
    A1 = relu1.forward(Z1)
    
    Z2 = layer_hidden.forward(A1)
    A2 = relu2.forward(Z2)
    
    y_pred = layer_output.forward(A2)
    
    # -- 2. Examen Evaluación Periódica --
    loss = MSELoss.forward(y_pred, y)
    history_numpy.append(loss)
    
    # -- 3. Retrospectiva Profunda (Backwards encadenado) --
    loss_grad = MSELoss.backward(y_pred, y)
    
    dA2 = layer_output.backward(loss_grad, lr=lr)
    
    dZ2 = relu2.backward(dA2, lr=lr)
    dA1 = layer_hidden.backward(dZ2, lr=lr)
    
    dZ1 = relu1.backward(dA1, lr=lr)
    _ = layer_input.backward(dZ1, lr=lr)
    
    if (epoch + 1) % 500 == 0:
        print(f"-> Época [{(epoch+1):04d}/{epochs}] - MSE Loss: {loss:.6f}")

# Gráfico Resultante Numpy
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_numpy, color='orange')
plt.title("Numpy Arquitectura Profunda")
plt.xlabel("Epochs")
plt.ylabel("Loss")

plt.subplot(1, 2, 2)
plt.scatter(X, y, color='cyan', alpha=0.3)
plt.plot(X, y_pred, color='orange', linewidth=3, label="Net_Numpy_Manual")
plt.title("Convergencia NumPy")
plt.legend()
plt.show()

---
## 2. PyTorch Low-Level: Delegando las Derivadas al Dios `Autograd`

Si te das cuenta, tuvimos que escribir `dW`, `db`, `dZ`, `dA`... imagináte construir ChatGPT escribiendo cada derivada a mano.

**La Mágia Industrial SOTA**: Usando PyTorch, instanciamos matrices crudas. Si activamos explícitamente `requires_grad=True`, PyTorch documentará matemáticamente en C++ por debajo todos estos pasos `dA2`, `dZ2`. Cuando tú llames a la única instrucción magistral `.backward()`, todo el grafo explota devolviendo la "culpa" a su matriz de origen al instante.

In [ ]:
# Base Tensorial PyTorch
X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

# CREACIÓN DE PARÁMETROS: Respetando las Tres Capas Reales.
W1 = torch.randn(1, 32, requires_grad=True)
b1 = torch.zeros(1, 32, requires_grad=True)

W2 = torch.randn(32, 32, requires_grad=True) # Los pesos oscilan del 32 oculto hacia un nuevo 32 oculto
b2 = torch.zeros(1, 32, requires_grad=True)

W3 = torch.randn(32, 1, requires_grad=True)  # El 32 oculto hacia la única respuesta continua terminal 1.
b3 = torch.zeros(1, 1, requires_grad=True)

epochs = 2000
lr = 0.02
history_torch_low = []

print(f"-> Training Loop Autograd Multi-Level: {epochs} Epochs...")
for epoch in range(epochs):
    # -- 1. PASADA FRONTAL (Invocando la capa oculta sin problema) --
    Z1 = X_t @ W1 + b1 
    A1 = torch.relu(Z1)
    
    Z2 = A1 @ W2 + b2
    A2 = torch.relu(Z2)
    
    y_pred_t = A2 @ W3 + b3
    
    # -- 2. FORMALIZACIÓN DE CULPA --
    loss = torch.mean((y_pred_t - y_t)**2)
    history_torch_low.append(loss.item())
    
    # -- 3. RETROPROPAGACIÓN NUCLEAR AUTOMÁTIZADA --
    loss.backward()  # Propaga al infinito rellenando los .grad pertinentes
    
    # -- 4. OPTIMIZACIÓN --
    with torch.no_grad():
        W1 -= lr * W1.grad
        b1 -= lr * b1.grad
        
        W2 -= lr * W2.grad
        b2 -= lr * b2.grad
        
        W3 -= lr * W3.grad
        b3 -= lr * b3.grad
        
        # ¡HIGIENE COMPUTACIONAL DE OBLIGADO CUMPLIMIENTO! 
        W1.grad.zero_()
        b1.grad.zero_()
        W2.grad.zero_()
        b2.grad.zero_()
        W3.grad.zero_()
        b3.grad.zero_()

    if (epoch + 1) % 500 == 0:
        print(f"-> Época [{(epoch+1):04d}/{epochs}] - Loss.item(): {loss.item():.6f}")

# Gráfico Resultante
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_torch_low, color='lime')
plt.title("Convergencia Tensor PyTorch")
plt.xlabel("Epochs")

plt.subplot(1, 2, 2)
plt.scatter(X, y, color='cyan', alpha=0.3)
with torch.no_grad():
    plt.plot(X, y_pred_t.numpy(), color='lime', linewidth=3, label="Net_Autograd")
plt.legend()
plt.show()

---
## 3. PyTorch Alta Ingeniería: API Orientada a Objetos (`nn.Module`)

En la industria el Ing. ML no declara tensores explícitamente y suscita bucles con `without.no_grad()` extensísimos. Delegamos la labor de instanciación completa en `torch.nn`.

In [ ]:
# EMPAQUETADO ARSQUITECTÓNICO DEL MODELO PROFESIONAL
model = nn.Sequential(
    nn.Linear(in_features=1, out_features=32),  # Frontal INPUT Layer
    nn.ReLU(),  
    nn.Linear(in_features=32, out_features=32), # Internal HIDDEN Layer (Aislada verdaderamente)
    nn.ReLU(), 
    nn.Linear(in_features=32, out_features=1)   # End OUTPUT Layer
)

print("Vista Panorámica del Grafo Ensamblado (Profundo):")
print(model)

# GESTIÓN DE EVALUADOR
criterion = nn.MSELoss()                           

# ASIGNACIÓN DEL OPTIMIZADOR SGD A TODOS LOS ENGRANAJES
optimizer = optim.SGD(model.parameters(), lr=0.02)  

epochs = 2000
history_sota = []

print(f"\nRutina Profesional en ejecución abstracta SOTA ({epochs} Epochs)...")
for epoch in range(epochs):
    # -- 1. INVOCACIÓN SÍNCRONA --
    y_pred = model(X_t)
    
    # -- 2. EVALUACIÓN Y LOSS --
    loss = criterion(y_pred, y_t)
    history_sota.append(loss.item())
    
    # -- LA TRINIDAD INQUEBRANTABLE DEL TRAINING LOOP MODERNO: --
    optimizer.zero_grad()  # Reset global de derivadas matriciales
    loss.backward()        # Propagación de fallos
    optimizer.step()       # Actualización en racimal SGD global de los Parameters

    if (epoch + 1) % 500 == 0:
        print(f"-> Época [{(epoch+1):04d}/{epochs}] - Loss SOTA: {loss.item():.6f}")

# Demostración SOTA Final
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_sota, color='magenta')
plt.title("PyTorch Profesional SOTA")
plt.xlabel("Epochs")
plt.ylabel("MSE Loss")

plt.subplot(1, 2, 2)
plt.scatter(X, y, color='cyan', alpha=0.3)

with torch.no_grad():
    plt.plot(X, model(X_t).numpy(), color='magenta', linewidth=3, label="DNN Model SOTA")
plt.title("Predicción Perfecta Oculta")
plt.legend()
plt.show()